# 🚁 Mission Tello : Exploration "Nez en Avant" + SLAM

Ce notebook exécute la version améliorée de l'exploration :
1.  **Scan Initial** : Rotation 360° au décollage.
2.  **Navigation Orientée** : Le drone pivote vers sa cible avant d'avancer (plus sûr pour la caméra frontale).
3.  **Scan Périodique** : Tous les 5 mètres parcourus, le drone s'arrête et scanne à 360°.
4.  **Cartographie SLAM** : Génération de la carte en temps réel.

In [ ]:
%matplotlib inline
import time
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Import des modules modifiés
from exploration import ExplorationMission, MissionConfig, MissionStatus
from visual_slam import VisualSLAM
from vision import VideoStream

print("✅ Modules chargés.")

### 1. Configuration de la Mission
Nous activons l'évitement et la cartographie. La zone est définie à 3m x 3m.

In [ ]:
config = MissionConfig(
    area_width=300,          
    area_height=300,
    exploration_altitude=100,
    step_size=50,            
    pattern="snake",         
    enable_avoidance=True,
    enable_mapping=True
)

# Initialisation (Mode Réel)
mission = ExplorationMission(config, simulation_mode=False)

print("Préparation...")
if mission.prepare_mission():
    print("✅ Drone connecté et prêt.")
    
    # Initialisation du SLAM
    video_stream = VideoStream(drone=mission.controller.drone, simulation_mode=False)
    slam = VisualSLAM(video_stream=video_stream, simulation_mode=False)
else:
    print("❌ Erreur connexion.")

### 2. Démarrage
Observez le comportement :
* Le drone va décoller.
* **Il fera immédiatement une rotation 360°.**
* Il s'orientera vers le premier point et avancera.
* Après 5m de trajet cumulé, il refera un scan.

In [ ]:
try:
    slam.start()
    time.sleep(2)
    
    print("🚀 DÉCOLLAGE ET EXPLORATION...")
    success = mission.start_exploration()
    
    if success:
        start_time = time.time()
        
        # Boucle de visualisation
        while mission.status == MissionStatus.IN_PROGRESS:
            # Données
            grid = slam.occupancy_grid
            pos = mission.controller.position
            dist_scan = mission.distance_since_last_scan
            
            # Affichage
            clear_output(wait=True)
            plt.figure(figsize=(12, 6))
            
            # Carte SLAM
            plt.subplot(1, 2, 1)
            plt.imshow(grid, cmap='magma', vmin=-1, vmax=100, origin='lower')
            plt.title(f"SLAM Map (Pos: {pos.x:.0f}, {pos.y:.0f})")
            
            # Infos Texte
            plt.subplot(1, 2, 2)
            plt.axis('off')
            info_text = (
                f"STATUS: {mission.status.value}\n"
                f"Temps: {time.time() - start_time:.1f}s\n"
                f"Dist. depuis scan: {dist_scan:.0f} / 500 cm\n"
                f"Waypoints: {mission.waypoints_completed}/{mission.total_waypoints}"
            )
            plt.text(0.1, 0.5, info_text, fontsize=14)
            
            plt.show()
            time.sleep(0.5)
            
        print("Mission terminée.")
        
except KeyboardInterrupt:
    print("⚠️ Arrêt d'urgence utilisateur")
    mission.emergency_stop()
finally:
    mission.stop_exploration()
    slam.stop()

In [ ]:
# Export des données
output_dir = "resultats_exploration_nose_first"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
mission.export_results(os.path.join(output_dir, "mission"))
slam.export_map(os.path.join(output_dir, "slam"))
print(f"Données exportées dans {output_dir}")